# EP05 — Naive Bayes Classifier
**COMPSCI 713 · S1 2024 Q5 · 11 marks**

| Part | Topic | Marks |
|------|-------|-------|
| a | Two simplifying (naive) assumptions | 6 |
| b | What additional info would full Bayes need? | 5 |

---

## Part a — The Two Naive Assumptions [6 marks]

**Assumption 1 — Conditional Feature Independence**

Given the class label (spam/not-spam), all features (words) are assumed *independent* of each other:

```
P(w1, w2, ..., wn | class) = P(w1|class) * P(w2|class) * ... * P(wn|class)
```

This is *naive* because in reality words are correlated — 'credit' and 'card' co-occur far more than chance predicts. NB ignores all pairwise and higher-order dependencies.

**Assumption 2 — Bag of Words (Position Independence)**

The order and position of words are ignored. Only word *presence* or *frequency* matters.
'Win prize now' and 'Now win prize' are treated identically.

This is naive because word order carries meaning ('not good' vs 'good not').

**Why still useful despite being naive?**
- Dramatically fewer parameters to estimate
- Works well in practice — especially for text classification
- Fast training and inference

---

## Part b — What Full Bayes Would Need [5 marks]

Without the independence assumption, the full Bayesian estimate requires:

```
P(spam | w1, w2, ..., wn) proportional to P(w1, w2, ..., wn | spam) * P(spam)
```

You must estimate the **full joint probability** P(w1, w2, ..., wn | class) over all word combinations.

**The problem:**
- With vocabulary size V, there are **2^V** possible word-presence patterns per message
- For V=10,000 words: 2^10000 possible events — astronomically more than any corpus contains
- You would need to store **all pairwise, triple, ..., n-way co-occurrence counts** for every combination
- This requires exponentially more training data and is computationally intractable

**In short:** you would need the full joint likelihood table — an exponential number of parameters that cannot be learned from any realistic training set.

---

## Code Demo — Naive Bayes Spam Classifier

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report
import numpy as np

emails = [
    'buy cheap offers now free',
    'win million dollars click here',
    'claim your free prize today',
    'urgent account suspended verify now',
    'limited offer act immediately free gift',
    'meeting tomorrow at 3pm conference room',
    'attached is the project report draft',
    'reminder team lunch on friday noon',
    'can we schedule a call this week',
    'the quarterly results are in the document',
    'free money transfer click now win',
    'please review the attached proposal',
]
labels = [1,1,1,1,1,0,0,0,0,0,1,0]  # 1=spam, 0=ham

# Assumption 2 (bag of words) is encoded by CountVectorizer
vec = CountVectorizer()
X = vec.fit_transform(emails)

# Assumption 1 (conditional independence) is encoded in the NB math
# P(email|spam) = P(w1|spam) * P(w2|spam) * ...  <- independence!
nb = MultinomialNB(alpha=1.0)  # alpha=Laplace smoothing
nb.fit(X, labels)

print(classification_report(labels, nb.predict(X), target_names=['Ham','Spam']))

# Top spam words
names = vec.get_feature_names_out()
top = np.argsort(nb.feature_log_prob_[1])[-8:]
print('Top spam indicator words:', list(names[top]))

# Demonstrate the independence assumption explicitly
print('\n--- Conditional independence in action ---')
test = vec.transform(['free prize click now'])
lp = nb.predict_log_proba(test)[0]
print(f'log P(ham|email)  = {lp[0]:.2f}')
print(f'log P(spam|email) = {lp[1]:.2f}')
print(f'Prediction: {"SPAM" if lp[1]>lp[0] else "HAM"}')
print('(This is the product of individual word likelihoods — the naive assumption at work)')